In [6]:
import sys
import numpy as np
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "kxor_code").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT_DIR = ROOT / "data" / "problem_instances"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Output directory:", OUT_DIR)

Project root: /Users/daphnejanissen/Documents/DSDM/Quantum Research Project/QuarticSpeedupK-XOR
Output directory: /Users/daphnejanissen/Documents/DSDM/Quantum Research Project/QuarticSpeedupK-XOR/data/problem_instances


In [7]:
m_values     = np.linspace(1, 1000, 1000, dtype=int).tolist()
n_values     = np.linspace(1, 500, 500, dtype=int).tolist()
k_values     = np.linspace(2, 10, 9, dtype=int).tolist()
ell_values   = np.linspace(2, 10, 5, dtype=int).tolist()
eps_values   = np.linspace(0.1, 0.3, 5).tolist()
kappa_values = np.linspace(0.5, 1.0, 5).tolist()

In [8]:
import math
from bisect import bisect_left
from math import comb
from kxor_code.problem_set_generation.kikuchi_matrix_generator import *

def brute_force_problem_parameter_search(
    n_values: list[int], k_values: list[int], ell_values: list[int],
    eps_values: list[float], kappa_values: list[float], m_values: list[int],
    n_factor: int = 100
):
    results = []
    m_sorted = sorted(m_values)

    for n in n_values:
        for k in k_values:
            for ell in ell_values:
                for eps in eps_values:
                    for kappa in kappa_values:

                        # ---- structural constraints (same as in check_alice_theorem) ----
                        if k <= 0 or k % 2 != 0:
                            continue
                        if ell < k / 2:
                            continue
                        if not (0 < kappa <= 1):
                            continue
                        if not (0 < eps <= kappa / (2 + kappa)):
                            continue
                        if n_factor is not None and n < n_factor * k * ell:
                            continue

                        # ---- compute the Delta threshold ----
                        C_kappa = (
                            2 * (1 + eps) * (1 + kappa) / (kappa ** 2)
                            * (1 / comb(k, k // 2))
                            * math.log(n)
                        )
                        delta_threshold = C_kappa * (n / ell) ** ((k - 2) / 2)

                        # Delta = m/n >= threshold  ->  m >= n * threshold
                        m_required = math.ceil(n * delta_threshold)

                        # ---- pick smallest allowed m that satisfies it ----
                        idx = bisect_left(m_sorted, m_required)
                        if idx == len(m_sorted):
                            continue
                        m = m_sorted[idx]

                        # ---- verify using your original theorem checker ----
                        r = check_alice_theorem(
                            ell=ell, n=n, k=k, kappa=kappa, eps=eps, m=m,
                            n_factor=n_factor
                        )

                        if not r["failure"]:
                            # IMPORTANT: add back the inputs so you can sort/print them
                            r.update({"n": n, "k": k, "ell": ell, "eps": eps, "kappa": kappa, "m": m})
                            results.append(r)

    return results


In [9]:
results = []

for nf in range(1, 4):  # 1,2,3
    r = brute_force_problem_parameter_search(
        n_values=n_values,
        k_values=k_values,
        ell_values=ell_values,
        eps_values=eps_values,
        kappa_values=kappa_values,
        m_values=m_values,
        n_factor=nf
    )
    results.extend(r)

print(f"Found {len(results)} feasible parameter sets (n_factor in [1,2,3])")




Found 15536 feasible parameter sets (n_factor in [1,2,3])


In [10]:
best = min(results, key=lambda r: (r["n"], r["m"]))

print("\nBEST (smallest n, m):")
print(f"n      = {best['n']}")
print(f"m      = {best['m']}")
print(f"k      = {best['k']}")
print(f"ell    = {best['ell']}")
print(f"kappa  = {best['kappa']}")
print(f"eps    = {best['eps']}")
print(f"Delta  = {best['Delta']:.4f}")
print(f"Delta_threshold = {best['Delta_threshold']:.4f}")
print(f"n / (k·ell) = {best['n_over_k_l']:.2f}")
print(f"Failure prob ≤ {best['failure_probability_bound']:.2e}")



BEST (smallest n, m):
n      = 4
m      = 13
k      = 2
ell    = 2
kappa  = 1.0
eps    = 0.1
Delta  = 3.2500
Delta_threshold = 3.0498
n / (k·ell) = 1.00
Failure prob ≤ 2.27e+00


In [11]:
import math

# Failure probability bound
P_MAX = 0.1  

good = [r for r in results if r["failure_probability_bound"] <= P_MAX]

if not good:
    print(f"No feasible sets with failure_probability_bound <= {P_MAX}.")
else:
    best = min(good, key=lambda r: (r["n"], r["m"], r["k"], r["ell"]))

    print(f"Found {len(good)} feasible sets with failure prob bound <= {P_MAX}\n")
    print("BEST (smallest n, m) under failure-prob constraint:")
    print(f"n      = {best['n']}")
    print(f"m      = {best['m']}")
    print(f"k      = {best['k']}")
    print(f"ell    = {best['ell']}")
    print(f"kappa  = {best['kappa']}")
    print(f"eps    = {best['eps']}")
    print(f"Delta  = {best['Delta']:.4f}")
    print(f"Delta_threshold = {best['Delta_threshold']:.4f}")
    print(f"n / (k·ell) = {best['n_over_k_l']:.2f}")
    print(f"Failure prob ≤ {best['failure_probability_bound']:.2e}")


Found 6879 feasible sets with failure prob bound <= 0.1

BEST (smallest n, m) under failure-prob constraint:
n      = 12
m      = 75
k      = 2
ell    = 6
kappa  = 1.0
eps    = 0.25
Delta  = 6.2500
Delta_threshold = 6.2123
n / (k·ell) = 1.00
Failure prob ≤ 7.22e-02


In [12]:
small_n_sets = [r for r in results if r["n"] < 31 and r["failure_probability_bound"] < 0.10]

print(f"Found {len(small_n_sets)} parameter sets with n < 31\n")

for i, r in enumerate(small_n_sets, 1):
    print(f"--- Set {i} ---")
    print(f"n      = {r['n']}")
    print(f"m      = {r['m']}")
    print(f"k      = {r['k']}")
    print(f"ell    = {r['ell']}")
    print(f"kappa  = {r['kappa']}")
    print(f"eps    = {r['eps']}")
    print(f"Delta  = {r['Delta']:.4f}")
    print(f"Delta_threshold = {r['Delta_threshold']:.4f}")
    print(f"n / (k·ell) = {r['n_over_k_l']:.2f}")
    print(f"Failure prob ≤ {r['failure_probability_bound']:.2e}")
    print()


Found 773 parameter sets with n < 31

--- Set 1 ---
n      = 12
m      = 116
k      = 2
ell    = 6
kappa  = 0.75
eps    = 0.25
Delta  = 9.6667
Delta_threshold = 9.6635
n / (k·ell) = 1.00
Failure prob ≤ 7.22e-02

--- Set 2 ---
n      = 12
m      = 92
k      = 2
ell    = 6
kappa  = 0.875
eps    = 0.25
Delta  = 7.6667
Delta_threshold = 7.6069
n / (k·ell) = 1.00
Failure prob ≤ 7.22e-02

--- Set 3 ---
n      = 12
m      = 75
k      = 2
ell    = 6
kappa  = 1.0
eps    = 0.25
Delta  = 6.2500
Delta_threshold = 6.2123
n / (k·ell) = 1.00
Failure prob ≤ 7.22e-02

--- Set 4 ---
n      = 12
m      = 95
k      = 2
ell    = 6
kappa  = 0.875
eps    = 0.3
Delta  = 7.9167
Delta_threshold = 7.9111
n / (k·ell) = 1.00
Failure prob ≤ 3.42e-02

--- Set 5 ---
n      = 12
m      = 78
k      = 2
ell    = 6
kappa  = 1.0
eps    = 0.3
Delta  = 6.5000
Delta_threshold = 6.4608
n / (k·ell) = 1.00
Failure prob ≤ 3.42e-02

--- Set 6 ---
n      = 13
m      = 130
k      = 2
ell    = 6
kappa  = 0.75
eps    = 0.25
Delta  = 

In [13]:
import pickle

with open("theorem_results.pkl", "wb") as f:
    pickle.dump(results, f)
